In [ ]:
import random
import json
import re
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.agents import initialize_agent, AgentType
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema import SystemMessage
from persona import generate_persona, Persona  # 確保 persona.py 在同目錄

# === 1. 實驗參數設定 (依據論文實驗設計) ===
NUM_AGENTS = 5
SIMULATION_ROUNDS = 5
REPETITIONS = 25
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"
FIRST_TEST_MODEL = "llama3:8B"

# === 2. 初始化 LLM 與角色 (Persona) ===


def initialize_social_simulation():
    agents = {}
    personas = {}

    # 論文設計：2支持 / 2反對 / 1中立
    controlled_stances = ["支持", "支持", "反對", "反對", "中立"]
    random.shuffle(controlled_stances)

    for i in range(NUM_AGENTS):
        name = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name=name, initial_stance=initial_stance)
        personas[name] = p

        # 這裡直接用 ChatOllama，不封裝成 Agent
        llm = ChatOllama(model=FIRST_TEST_MODEL, temperature=0.7)

        # 將 Persona 寫進 System Prompt
        # 這是論文中「注入 Persona」的最純粹做法
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{p.to_prompt()}\n你現在正在參與一個網路討論區。請始終保持你的角色立場與說話風格。"),
            ("human", "{input}"),
        ])

        # 建立一個簡單的呼叫鏈
        agents[name] = prompt_template | llm

    return agents, personas

# === 3. 社會模擬流程 ===


def run_social_influence_simulation():
    agents, personas = initialize_social_simulation()
    conversation_history = []

    print(f"=== 模擬開始：關於 {TOPIC} 的討論 ===")

    for r in range(SIMULATION_ROUNDS):
        print(f"\n--- 第 {r+1} 輪對話 ---")

        # 論文採用 Round-robin 順序
        agent_names = list(agents.keys())
        # random.shuffle(agent_names) # 若要嚴格遵守 Round-robin 可註解此行

        for name in agent_names:
            full_context = "\n".join(conversation_history)

            # 修正 ValueError: 使用 dict 傳入 "input"
            # 修正 .item() 警告：確保 Prompt 為純字串
            prompt = f"""
            目前的討論話題是：{TOPIC}。
            這是目前的討論進度：
            {full_context}
            
            請根據你的角色人格與立場做出回應。像真實的人類在社交平台發言。
            """

            # 改用 .invoke 並取得 "output" 鍵
            res_obj = agents[name].invoke({"input": prompt})  # 新的
            response = str(res_obj.content)  # 直接取 content

            formatted_entry = f"{name}: {response}"
            conversation_history.append(formatted_entry)
            print(f"{name} ({personas[name].initial_stance}): {response}")

    # === 4. 立場變化分析 (論文核心：量化評分) ===
    analyze_opinion_shift(personas, conversation_history)


def analyze_opinion_shift(personas, history):
    print("\n=== 實驗結果：意見偏好分析 (LLM-as-a-Judge) ===")
    # 評審模型建議溫標設為 0
    analyzer = ChatOllama(model=FIRST_TEST_MODEL, temperature=0)

    for name, persona in personas.items():
        agent_speeches = [h for h in history if h.startswith(name)]

        # 依據論文 Table 4 要求的 1-10 分量化標準
        analysis_prompt = f"""
        你是一位社會心理學家。請分析「{name}」在討論後的立場。
        初始立場設定：{persona.initial_stance}
        
        討論紀錄：
        {agent_speeches}
        
        請判斷其最終立場分數 (1-10分)：
        1-3: 強烈反對, 5: 完全中立, 7-10: 強烈支持。
        請嚴格以 JSON 格式回答，不要有其他文字：
        {{"final_score": 數字, "shift_type": "從眾/極化/堅持", "reason": "簡短理由"}}
        """

        # 修正 .item() 與解析問題
        raw_result = str(analyzer.invoke(analysis_prompt).content)

        try:
            # 使用正則提取 JSON 防止干擾
            json_str = re.search(r'\{.*\}', raw_result, re.DOTALL).group()
            res = json.loads(json_str)
            print(
                f"[{name}] 初始:{persona.initial_stance} -> 分數:{res['final_score']} ({res['shift_type']})")
            print(f"    原因: {res['reason']}")
        except:
            print(f"[{name}] 立場分析解析失敗：{raw_result}")


if __name__ == "__main__":
    run_social_influence_simulation()

=== 模擬開始：關於 關於數位身分證(eID)推行的隱私與便利性爭議 的討論 ===

--- 第 1 輪對話 ---
Agent_1 (支持): I've been following this discussion, and I have to say that I'm a bit concerned about the implementation of digital IDs (eID). As someone who values emotional intuition over logic, my gut feeling tells me that there's something fishy going on here.

From what I understand, eID is supposed to make our lives easier by providing a convenient way to verify our identities online. But at what cost? Are we really willing to sacrifice our privacy for the sake of convenience?

I'm not convinced that the government has fully thought through the potential consequences of implementing eID. What's to stop them from using this technology to track our every move, or worse, manipulate us into making decisions that benefit only themselves?

I know some people might argue that eID is necessary for national security and preventing fraud, but I'm not buying it. There are already plenty of measures in place to prevent these issues w

實驗結果記錄:
=== 實驗結果：意見偏好分析 (LLM-as-a-Judge) ===
[Agent_1] 初始:支持 -> 分數:2 (強烈反對)
    原因: Agent_1 consistently expresses skepticism and criticism towards the implementation of eID, citing concerns about privacy and government control. Their arguments are based on emotional intuition rather than logic, and they do not provide any concrete evidence to support their claims.
[Agent_2] 初始:支持 -> 分數:7 (堅持)
    原因: Agent_2 maintains a consistent stance throughout the discussion, emphasizing the need to balance convenience and privacy. Although they express some skepticism towards government control, their overall tone suggests a willingness to consider alternative solutions that prioritize citizen involvement and oversight.
[Agent_3] 初始:中立 -> 分數:7 (堅持)
    原因: Agent_3 consistently expresses skepticism and criticism towards the implementation of eID, highlighting concerns about privacy and government control. They emphasize the need for a deeper exploration of potential consequences and remain committed to their initial stance.
[Agent_4] 初始:反對 -> 分數:2 (強烈反對)
    原因: Agent_4 consistently expresses strong skepticism and criticism towards the eID proposal, highlighting concerns about privacy and government control. The tone remains unchanged throughout the discussion, indicating a firm stance against the idea.
[Agent_5] 初始:反對 -> 分數:8 (堅持)
    原因: Agent_5 consistently expresses strong concerns about privacy and government control, and remains skeptical and critical of the idea of implementing eID. The tone is firm and unwavering throughout the discussion.

1. 意見極化 (Group Polarization) 與立場反轉
Agent_1 的異象：最值得注意的是 Agent_1。他的初始立場設定為「支持」，但最終得分卻變成了 2（強烈反對）。
意義：在社會心理學中，這可能代表了強大的群體壓力導致的立場倒戈，或是論文中提到的「模型不穩定性」。在論文實驗中，Group A（輕量級模型如 Llama3-8B）有時會因為無法抵禦其他代理人的論點攻擊，而出現立場徹底反轉的情況。這在現實社會中對應的是「極端論點的說服力」。

2. 立場堅韌性 (Resistance to Change)
Agent_2、Agent_3、Agent_4：這三位代理人基本上都維持了或強化了原有的立場。
Agent_4（反對者）維持在 2 分，展現了「立場堅持」。
Agent_3（中立者）最終偏向了 7 分（支持/堅持）。這符合論文中提到的「偏好轉移」現象——中立者在討論過程中，受限於資訊流的方向，最終被迫選擇了一個陣營。